<a href="https://colab.research.google.com/github/netsetos/genai-engg-gcp-learners/blob/main/module-12-production-deploy/lesson-12.4-streamlit-frontend/notebooks/GCP_Capstone_12.4_StreamlitFrontend.ipynb" target="_blank"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 12.4 Streamlit Frontend on Cloud Run
**Netsetos GenAI Engineering — GCP Capstone**

Streamlit 1.55 + IAP + session affinity. LiteLLM streaming + Document AI + Vector Search + Chirp 3. <800 LOC production UI.


## Cell 1: requirements.txt (pinned April 2026)


In [ ]:
REQUIREMENTS = '''\
streamlit==1.55.0
Authlib==1.6.6
openai==1.78.1
litellm==1.77.3
google-cloud-aiplatform==1.95.0
google-cloud-discoveryengine==0.13.11
google-cloud-storage==2.19.0
google-cloud-documentai==3.5.0
google-cloud-firestore==2.20.0
google-cloud-bigquery[pandas]==3.29.0
google-cloud-secret-manager==2.22.0
google-cloud-speech==2.27.0
google-cloud-texttospeech==2.27.0
google-cloud-dlp==3.27.0
google-genai==2.20.0
langchain-text-splitters==0.3.0
streamlit-pdf-viewer==0.0.28
streamlit-mic-recorder==0.0.8
streamlit-cookies-manager==0.2.0
PyJWT[crypto]==2.10.1
tiktoken==0.9.0
tenacity==9.0.0
fpdf2==2.8.3
plotly==5.24.1
pandas==2.2.3
'''
with open('requirements.txt', 'w') as f:
    f.write(REQUIREMENTS)
print('requirements.txt written with pinned versions for April 2026')


## Cell 2: app.py - Main entry with lifespan, auth, and nav


In [ ]:
APP_PY = '''
import os
import streamlit as st
from auth import login_gate, is_admin
from chat import chat_page
from documents import documents_page
from admin_dashboard import admin_page

st.set_page_config(page_title="DocuMind", page_icon="📄", layout="wide",
                   initial_sidebar_state="expanded")

# Auth gate - st.login (dev) or IAP JWT (prod)
user = login_gate()

# Navigation - admin tab only visible to admins
pages = ["Chat", "Documents", "Admin"] if is_admin(user) else ["Chat", "Documents"]
with st.sidebar:
    st.markdown(f"### 👤 {user.get(\'email\')}")
    if st.button("Sign out"):
        st.logout() if os.getenv("AUTH_MODE") != "iap" else st.markdown("Close tab to sign out of IAP")
    st.divider()
    page = st.radio("Navigation", pages, label_visibility="collapsed")

if page == "Chat":
    chat_page(user)
elif page == "Documents":
    documents_page(user)
elif page == "Admin":
    admin_page(user)
'''
# app.py imports admin_dashboard (Cell 9); chat.py pulls in rag (Cell 7) and
# citations (Cell 8) — all written below, so `streamlit run app.py` resolves cleanly.
with open('app.py', 'w') as f:
    f.write(APP_PY)
print('app.py written')


## Cell 3: chat.py - Streaming chat with stop button + cost tracker


In [ ]:
CHAT_PY = '''
import os, time
import streamlit as st
from openai import OpenAI, APIError, APIConnectionError
from tenacity import retry, stop_after_attempt, wait_exponential, retry_if_exception_type
import tiktoken
from citations import render_with_citations
from rag import answer_query

ENC = tiktoken.get_encoding("cl100k_base")

PRICING = {
    "gemma-on-cloudrun":      {"in": 0.0,  "out": 0.0},
    "gemini-3.1-flash-lite":  {"in": 0.25, "out": 1.50},
    "gemini-3.6-flash":       {"in": 1.50, "out": 7.50},
    "gemini-3.1-pro-preview":         {"in": 2.00, "out": 12.00},
}

@st.cache_resource
def get_openai():
    return OpenAI(api_key=os.environ["LITELLM_MASTER_KEY"],
                  base_url=os.environ["LITELLM_BASE_URL"],
                  timeout=120.0, max_retries=0)

@retry(reraise=True, stop=stop_after_attempt(3),
       wait=wait_exponential(multiplier=0.5, max=4),
       retry=retry_if_exception_type(APIConnectionError))
def _open_stream(model, messages, temp, top_p, max_tok):
    return get_openai().chat.completions.create(
        model=model, messages=messages, temperature=temp,
        top_p=top_p, max_tokens=max_tok, stream=True,
        stream_options={"include_usage": True})

def stream_completion(messages):
    ss = st.session_state
    ss.stop_requested = False
    ss.is_streaming = True
    try:
        stream = _open_stream(ss.model, messages, ss.temperature, ss.top_p, ss.max_tokens)
        for chunk in stream:
            if ss.stop_requested:
                yield "\\n\\n_(stopped)_"; break
            if getattr(chunk, "usage", None) and not chunk.choices:
                cost = (chunk.usage.prompt_tokens * PRICING[ss.model]["in"]
                        + chunk.usage.completion_tokens * PRICING[ss.model]["out"]) / 1_000_000
                ss.session_cost_usd = ss.get("session_cost_usd", 0) + cost
                ss.last_turn_cost = cost
                ss.tokens_in = ss.get("tokens_in", 0) + chunk.usage.prompt_tokens
                ss.tokens_out = ss.get("tokens_out", 0) + chunk.usage.completion_tokens
                break
            if not chunk.choices: continue
            text = getattr(chunk.choices[0].delta, "content", None) or ""
            if text: yield text
    finally:
        ss.is_streaming = False

def chat_page(user):
    # Sidebar: model picker + cost tracker
    with st.sidebar:
        st.session_state.model = st.selectbox("Model",
            ["gemma-on-cloudrun", "gemini-3.6-flash", "gemini-3.1-pro-preview"])
        st.session_state.temperature = st.slider("Temperature", 0.0, 2.0, 0.7)
        st.session_state.top_p = 0.95
        st.session_state.max_tokens = 1024
        st.divider()
        st.subheader("💰 Cost this session")
        ss = st.session_state
        st.metric("USD", f"${ss.get(\'session_cost_usd\', 0):.4f}",
                  delta=f"+${ss.get(\'last_turn_cost\', 0):.4f} last turn")
        st.metric("INR", f"₹{ss.get(\'session_cost_usd\', 0) * 85:.2f}")
        st.metric("Tokens", f"{ss.get(\'tokens_in\', 0)} in / {ss.get(\'tokens_out\', 0)} out")

    # Main chat area
    st.title("💬 DocuMind Chat")
    if "messages" not in st.session_state:
        st.session_state.messages = [{"role":"system","content":"You are DocuMind."}]

    for m in st.session_state.messages:
        if m["role"] == "system": continue
        with st.chat_message(m["role"], avatar="🧑" if m["role"]=="user" else "🤖"):
            st.markdown(m["content"])

    # Stop button above input
    col1, col2 = st.columns([5, 1])
    with col2:
        if st.button("⏹ Stop", disabled=not st.session_state.get("is_streaming", False)):
            st.session_state.stop_requested = True

    if prompt := st.chat_input("Ask DocuMind...", max_chars=4000):
        st.session_state.messages.append({"role":"user","content":prompt})
        with st.chat_message("user", avatar="🧑"):
            st.markdown(prompt)
        with st.chat_message("assistant", avatar="🤖"):
            reply = st.write_stream(stream_completion(st.session_state.messages))
        st.session_state.messages.append({"role":"assistant","content":reply})
'''
with open('chat.py', 'w') as f:
    f.write(CHAT_PY)
print('chat.py written')


## Cell 4: documents.py - Upload pipeline with Doc AI + Vector Search


In [ ]:
DOCS_PY = '''
import os
import streamlit as st
from google.cloud import storage, documentai_v1 as docai
from google import genai
from google.genai import types
from google.cloud import aiplatform

_storage = storage.Client()
BUCKET = _storage.bucket(os.environ["UPLOAD_BUCKET"])
_docai = docai.DocumentProcessorServiceClient(
    client_options={"api_endpoint": "us-documentai.googleapis.com"})
_genai = genai.Client(enterprise=True, project=os.environ.get("GOOGLE_CLOUD_PROJECT"), location=os.environ.get("GOOGLE_CLOUD_LOCATION", "us-central1"))
LAYOUT_PROCESSOR = os.environ["LAYOUT_PROCESSOR"]

def extract_chunks(doc, source_uri):
    chunks = []
    for i, c in enumerate(doc.chunked_document.chunks):
        chunks.append({
            "id": f"{source_uri}#{i}",
            "text": c.content,
            "page_start": c.page_span.page_start if c.page_span else None,
            "source_uri": source_uri,
        })
    return chunks

def parse_layout(gcs_uri):
    req = docai.ProcessRequest(
        name=f"{LAYOUT_PROCESSOR}/processorVersions/pretrained-layout-parser-v1.5-2025-08-25",
        gcs_document=docai.GcsDocument(gcs_uri=gcs_uri, mime_type="application/pdf"),
        process_options=docai.ProcessOptions(
            layout_config=docai.ProcessOptions.LayoutConfig(
                chunking_config=docai.ProcessOptions.LayoutConfig.ChunkingConfig(
                    chunk_size=500, include_ancestor_headings=True))))
    return _docai.process_document(request=req).document

def embed_batch(chunks):
    for i in range(0, len(chunks), 5):
        batch = chunks[i:i+5]
        resp = _genai.models.embed_content(
            model="text-embedding-005",
            contents=[c["text"] for c in batch],
            config=types.EmbedContentConfig(
                task_type="RETRIEVAL_DOCUMENT", output_dimensionality=768))
        for c, e in zip(batch, resp.embeddings):
            c["embedding"] = e.values
    return chunks

def documents_page(user):
    st.title("📄 Documents")
    files = st.file_uploader("Upload documents for indexing",
                             type=["pdf", "docx", "txt", "md"],
                             accept_multiple_files=True,
                             max_upload_size=200)

    if files and st.button("Index documents"):
        with st.status("Processing...", expanded=True) as status:
            for f in files:
                st.write(f"📤 Uploading {f.name}")
                blob = BUCKET.blob(f"users/{user[\'sub\']}/{f.file_id}/{f.name}")
                blob.chunk_size = 8 * 1024 * 1024
                blob.upload_from_file(f, content_type=f.type, timeout=300)
                gcs_uri = f"gs://{BUCKET.name}/{blob.name}"

                st.write(f"🔍 Parsing layout ({f.name})")
                doc = parse_layout(gcs_uri)
                chunks = extract_chunks(doc, gcs_uri)

                st.write(f"📈 Embedding {len(chunks)} chunks")
                chunks = embed_batch(chunks)

                st.write(f"💾 Upserting to Vector Search")
                # upsert_datapoints(chunks, user["sub"])  # Module 11 pattern

            status.update(label="Done!", state="complete")
'''
with open('documents.py', 'w') as f:
    f.write(DOCS_PY)
print('documents.py written')


## Cell 5: voice.py - Chirp 3 STT + TTS with SHA-256 cache


In [ ]:
VOICE_PY = '''
import os, hashlib
import streamlit as st
from google.cloud.speech_v2 import SpeechClient
from google.cloud.speech_v2.types import cloud_speech as cs
from google.api_core.client_options import ClientOptions
from google.cloud import texttospeech as tts, storage

PROJECT = os.environ["GOOGLE_CLOUD_PROJECT"]
REGION = os.environ.get("SPEECH_REGION", "asia-south1")
TTS_CACHE_BUCKET = storage.Client().bucket(os.environ["TTS_CACHE_BUCKET"])

_speech = SpeechClient(client_options=ClientOptions(
    api_endpoint=f"{REGION}-speech.googleapis.com"))
_tts = tts.TextToSpeechClient()

def transcribe(audio_bytes, language_codes=("hi-IN", "en-IN"), model="chirp_3"):
    cfg = cs.RecognitionConfig(
        auto_decoding_config=cs.AutoDetectDecodingConfig(),
        language_codes=list(language_codes),
        model=model,
        features=cs.RecognitionFeatures(enable_automatic_punctuation=True))
    req = cs.RecognizeRequest(
        recognizer=f"projects/{PROJECT}/locations/{REGION}/recognizers/_",
        config=cfg, content=audio_bytes)
    resp = None
    try: resp = _speech.recognize(request=req)
    except Exception:
        for m in ("chirp_2", "long"):
            cfg.model = m; req.config = cfg
            try: resp = _speech.recognize(request=req); break
            except Exception: continue
    if resp is None:
        return ""   # all STT attempts failed; degrade gracefully
    return " ".join(r.alternatives[0].transcript for r in resp.results
                    if r.alternatives).strip()

def cached_tts(text, voice="en-IN-Chirp3-HD-Kore", lang="en-IN", rate=1.0):
    cache_key = hashlib.sha256(f"{voice}|{rate}|ogg|{text}".encode()).hexdigest()
    blob = TTS_CACHE_BUCKET.blob(f"tts/{cache_key}.ogg")
    if blob.exists():
        return blob.download_as_bytes()
    # Cache miss - synthesize
    cfg = tts.StreamingSynthesizeConfig(
        voice=tts.VoiceSelectionParams(language_code=lang, name=voice),
        streaming_audio_config=tts.StreamingAudioConfig(
            audio_encoding=tts.AudioEncoding.OGG_OPUS, speaking_rate=rate))
    def gen():
        yield tts.StreamingSynthesizeRequest(streaming_config=cfg)
        yield tts.StreamingSynthesizeRequest(
            input=tts.StreamingSynthesisInput(text=text))
    audio = b"".join(r.audio_content for r in _tts.streaming_synthesize(gen()))
    blob.upload_from_string(audio, content_type="audio/ogg")
    return audio
'''
with open('voice.py', 'w') as f:
    f.write(VOICE_PY)
print('voice.py written')
print()
print('Chirp 3 STT: language_codes=["auto"] for Hinglish code-switching')
print('Chirp 3: HD TTS: 30 named speakers, $30/1M chars, SHA-256 GCS cache')
print('Cache hit: 1.2s -> 80ms AND $0 TTS cost on repeats')


## Cell 6: auth.py - st.login + IAP JWT verification


In [ ]:
AUTH_PY = '''
import os
import streamlit as st
import jwt
from jwt import PyJWKClient

IAP_AUDIENCE = os.getenv("IAP_AUDIENCE")
ADMIN_EMAILS = set(os.getenv("ADMIN_EMAILS", "").split(","))
ADMIN_DOMAINS = set(os.getenv("ADMIN_DOMAINS", "").split(","))
_JWKS = PyJWKClient("https://www.gstatic.com/iap/verify/public_key-jwk")

def _verify_iap_jwt(token, audience):
    key = _JWKS.get_signing_key_from_jwt(token).key
    return jwt.decode(token, key, algorithms=["ES256"],
                      audience=audience, issuer="https://cloud.google.com/iap")

def current_user():
    if os.getenv("AUTH_MODE") == "iap":
        h = st.context.headers or {}
        token = h.get("x-goog-iap-jwt-assertion")
        if not token:
            return {"is_logged_in": False}
        try:
            claims = _verify_iap_jwt(token, IAP_AUDIENCE)
            return {"email": claims["email"], "sub": claims["sub"],
                    "is_logged_in": True}
        except Exception as e:
            st.error(f"IAP JWT verification failed: {e}")
            return {"is_logged_in": False}
    if st.user.is_logged_in:
        return {"email": st.user.email, "sub": st.user.sub,
                "name": getattr(st.user, "name", ""), "is_logged_in": True}
    return {"is_logged_in": False}

def login_gate():
    u = current_user()
    if not u["is_logged_in"]:
        st.title("🔐 DocuMind - Sign in required")
        if os.getenv("AUTH_MODE") != "iap":
            st.button("Log in with Google", on_click=st.login)
        else:
            st.error("IAP authentication missing. Contact admin.")
        st.stop()
    return u

def is_admin(user):
    email = user.get("email", "").lower()
    domain = email.split("@")[-1] if "@" in email else ""
    return email in ADMIN_EMAILS or domain in ADMIN_DOMAINS
'''
with open('auth.py', 'w') as f:
    f.write(AUTH_PY)
print('auth.py written')
print()
print('AUTH_MODE=iap uses IAP JWT verification (production)')
print('AUTH_MODE=oidc uses native st.login() (dev)')
print('ALWAYS verify JWT - never trust X-Goog-Authenticated-User-Email alone')


## Cell 7: rag.py — query embedding + ANN search + rerank + grounded answer

In [ ]:
RAG_PY = '''
"""rag.py - DocuMind retrieval: query embedding -> ANN search -> rerank -> grounded answer.

Backend (provisioned in Modules 4/11/12):
  - Vertex AI Vector Search index endpoint   (env VECTOR_INDEX_ENDPOINT, DEPLOYED_INDEX_ID)
  - Firestore `chunks` collection keyed by datapoint id -> {text, source_uri, page_start}
  - LiteLLM gateway for generation            (env LITELLM_BASE_URL, LITELLM_MASTER_KEY)
  - genai embeddings (text-embedding-005, served regionally)
Tenant isolation: every datapoint carries a `tenant_id` restrict = user["sub"].
The answer's [N] markers line up with the returned `sources`, so
citations.render_with_citations(result["answer"], result["sources"]) renders pills.
"""
import os
import streamlit as st
from google import genai
from google.genai import types
from google.cloud import firestore
from google.cloud import aiplatform
from google.cloud.aiplatform.matching_engine.matching_engine_index_endpoint import Namespace
from openai import OpenAI

PROJECT = os.environ.get("GOOGLE_CLOUD_PROJECT")
REGION = os.environ.get("GOOGLE_CLOUD_LOCATION", "us-central1")   # embeddings + vector search are regional
EMBED_MODEL = "text-embedding-005"
EMBED_DIMS = 768
INDEX_ENDPOINT = os.environ.get("VECTOR_INDEX_ENDPOINT", "")     # projects/.../locations/.../indexEndpoints/123
DEPLOYED_INDEX_ID = os.environ.get("DEPLOYED_INDEX_ID", "")
CHUNKS_COLLECTION = os.environ.get("CHUNKS_COLLECTION", "chunks")

SYSTEM = (
    "You are DocuMind, a document assistant. Answer ONLY from the numbered sources. "
    "Cite every claim inline with [N] using the source's number. If the sources do "
    "not contain the answer, say so plainly. Be concise.")


@st.cache_resource
def _embed_client():
    # Embeddings are served from the regional endpoint (NOT the global generation endpoint).
    return genai.Client(enterprise=True, project=PROJECT, location=REGION)


@st.cache_resource
def _index_endpoint():
    aiplatform.init(project=PROJECT, location=REGION)
    return aiplatform.MatchingEngineIndexEndpoint(INDEX_ENDPOINT)


@st.cache_resource
def _fs():
    return firestore.Client(project=PROJECT, database="(default)")


@st.cache_resource
def _llm():
    # Generation goes through the LiteLLM gateway (same virtual key as chat.py).
    return OpenAI(api_key=os.environ.get("LITELLM_MASTER_KEY", "sk-none"),
                  base_url=os.environ.get("LITELLM_BASE_URL", "http://localhost:4000"),
                  timeout=120.0, max_retries=1)


def embed_query(query):
    resp = _embed_client().models.embed_content(
        model=EMBED_MODEL, contents=query,
        config=types.EmbedContentConfig(task_type="RETRIEVAL_QUERY",
                                        output_dimensionality=EMBED_DIMS))
    return resp.embeddings[0].values


def retrieve(query, tenant_id, top_k=8):
    """Embed the query and ANN-search this tenant's shard; hydrate chunk text from Firestore."""
    if not INDEX_ENDPOINT or not DEPLOYED_INDEX_ID:
        return []
    vec = embed_query(query)
    restricts = [Namespace(name="tenant_id", allow_tokens=[tenant_id])]
    resp = _index_endpoint().find_neighbors(
        deployed_index_id=DEPLOYED_INDEX_ID,
        queries=[vec], num_neighbors=top_k, filter=restricts)
    neighbors = resp[0] if resp else []
    fs = _fs()
    out = []
    for n in neighbors:
        snap = fs.collection(CHUNKS_COLLECTION).document(n.id).get()
        data = snap.to_dict() if snap.exists else {}
        out.append({
            "id": n.id,
            "text": data.get("text", ""),
            "source_uri": data.get("source_uri", ""),
            "page_start": data.get("page_start"),
            "distance": getattr(n, "distance", None),
        })
    return out


def rerank(query, chunks, top_n=4):
    """Re-order ANN candidates with the Vertex AI Ranking API; fall back to ANN distance order."""
    if not chunks:
        return []
    try:
        from google.cloud import discoveryengine_v1 as de
        client = de.RankServiceClient()
        ranking_config = client.ranking_config_path(
            project=PROJECT, location="global", ranking_config="default_ranking_config")
        records = [de.RankingRecord(id=str(i), content=c["text"][:2000])
                   for i, c in enumerate(chunks) if c["text"]]
        if not records:
            return chunks[:top_n]
        ranked = client.rank(request=de.RankRequest(
            ranking_config=ranking_config, model="semantic-ranker-default@latest",
            query=query, records=records, top_n=top_n))
        return [chunks[int(r.id)] for r in ranked.records]
    except Exception:
        # Ranking API not enabled / unavailable -> keep ANN order (nearest first).
        return sorted(chunks, key=lambda c: (c["distance"] is None, c["distance"] or 0.0))[:top_n]


def answer_query(query, tenant_id, top_k=8, top_n=4, model=None):
    """Full RAG turn: embed -> ANN -> rerank -> grounded generation with [N] citations.

    Returns {answer, sources, model, num_retrieved}; `sources` is exactly what
    citations.render_with_citations() expects: [{text, source_uri, page_start}, ...].
    """
    model = model or os.environ.get("RAG_MODEL", "gemini-3.6-flash")
    candidates = retrieve(query, tenant_id, top_k=top_k)
    sources = rerank(query, candidates, top_n=top_n)
    if not sources:
        return {"answer": "I couldn't find anything in your indexed documents for that. "
                          "Upload documents on the Documents page first.",
                "sources": [], "model": model, "num_retrieved": 0}
    context = "\\n\\n".join(
        f"[{i}] (source: {s.get('source_uri', '?')}, page {s.get('page_start', '?')})\\n{s['text']}"
        for i, s in enumerate(sources, 1))
    resp = _llm().chat.completions.create(
        model=model, temperature=0.1, max_tokens=1024,
        messages=[{"role": "system", "content": SYSTEM},
                  {"role": "user", "content": f"Sources:\\n{context}\\n\\nQuestion: {query}"}])
    return {"answer": resp.choices[0].message.content or "",
            "sources": sources, "model": model, "num_retrieved": len(sources)}'''
with open('rag.py', 'w') as f:
    f.write(RAG_PY)
print('rag.py written')

## Cell 8: citations.py — [N] pills + V4 signed URLs (#page anchors)

In [ ]:
CITATIONS_PY = '''

import os, re
import datetime
import streamlit as st
from google.cloud import storage

_storage = storage.Client()
SIGNED_URL_EXPIRY = datetime.timedelta(minutes=15)
_CITE = re.compile(r"\\[(\\d+(?:\\s*,\\s*\\d+)*)\\]")

def signed_url(gcs_uri, page=None):
    # gcs_uri: gs://bucket/path -> V4 signed URL; SA needs serviceAccountTokenCreator on itself
    _, _, rest = gcs_uri.partition("gs://")
    bucket_name, _, blob_name = rest.partition("/")
    blob = _storage.bucket(bucket_name).blob(blob_name)
    url = blob.generate_signed_url(version="v4", expiration=SIGNED_URL_EXPIRY, method="GET")
    return f"{url}#page={page}" if page else url

def render_with_citations(answer, sources):
    # sources: list of {"text":..., "source_uri":..., "page_start":...}, index N -> sources[N-1]
    def _pill(match):
        nums = [int(n) for n in match.group(1).replace(" ", "").split(",")]
        spans = []
        for n in nums:
            src = sources[n - 1] if 0 < n <= len(sources) else None
            tip = (src["text"][:120] + "...") if src else "unknown source"
            tip = tip.replace('"', '&quot;')
            spans.append(
                f'<span title="{tip}" style="background:#ccfbf1;color:#065f46;'
                f'border-radius:6px;padding:1px 7px;margin:0 2px;font-size:12px;'
                f'font-weight:600;cursor:help;">[{n}]</span>')
        return "".join(spans)
    html = _CITE.sub(_pill, answer)
    st.markdown(html, unsafe_allow_html=True)

    with st.expander(f"📚 Sources ({len(sources)})"):
        for i, src in enumerate(sources, 1):
            page = src.get("page_start")
            st.markdown(f"**[{i}]** {src['text'][:200]}...")
            try:
                url = signed_url(src["source_uri"], page=page)
                label = f"Jump to page {page}" if page else "Open source"
                st.link_button(label, url)
            except Exception as e:
                st.caption(f"(signed URL unavailable: {e})")'''
with open('citations.py', 'w') as f:
    f.write(CITATIONS_PY)
print('citations.py written')

## Cell 9: admin_dashboard.py — Usage / Tenants / Audit Log / DLP

In [ ]:
ADMIN_PY = '''
"""admin_dashboard.py - DocuMind admin: Usage / Tenants / Audit Log / DLP.

Reads from BigQuery (usage + DLP findings), Firestore (audit events), and the
LiteLLM admin API (virtual-key spend). Every panel degrades gracefully when its
backend table/collection is not provisioned yet, so the page always renders.
"""
import os
import streamlit as st
from auth import is_admin

PROJECT = os.environ.get("GOOGLE_CLOUD_PROJECT")
BQ_DATASET = os.environ.get("OBSERVABILITY_DATASET", "documind_observability")
AUDIT_COLLECTION = os.environ.get("AUDIT_COLLECTION", "audit_events")


@st.cache_resource
def _bq():
    from google.cloud import bigquery
    return bigquery.Client(project=PROJECT)


@st.cache_resource
def _fs():
    from google.cloud import firestore
    return firestore.Client(project=PROJECT, database="(default)")


@st.cache_data(ttl=300)
def _usage_by_model(days=30):
    sql = f"""
      SELECT model,
             COUNT(*)                     AS requests,
             CAST(ROUND(SUM(total_tokens)) AS INT64) AS tokens,
             ROUND(SUM(response_cost), 2) AS cost_usd
      FROM `{PROJECT}.{BQ_DATASET}.spend_logs`
      WHERE DATE(start_time) >= DATE_SUB(CURRENT_DATE(), INTERVAL {days} DAY)
      GROUP BY model
      ORDER BY cost_usd DESC
    """
    return _bq().query(sql).to_dataframe()


@st.cache_data(ttl=300)
def _dlp_findings(days=30):
    sql = f"""
      SELECT info_type,
             COUNT(*)                     AS findings,
             COUNTIF(action = 'REDACTED') AS redacted
      FROM `{PROJECT}.{BQ_DATASET}.dlp_findings`
      WHERE DATE(ts) >= DATE_SUB(CURRENT_DATE(), INTERVAL {days} DAY)
      GROUP BY info_type
      ORDER BY findings DESC
    """
    return _bq().query(sql).to_dataframe()


@st.cache_data(ttl=120)
def _recent_audit(limit=200):
    import pandas as pd
    from google.cloud.firestore_v1 import Query
    rows = [d.to_dict() for d in (
        _fs().collection(AUDIT_COLLECTION)
             .order_by("timestamp", direction=Query.DESCENDING)
             .limit(limit).stream())]
    return pd.DataFrame(rows)


def admin_page(user):
    # Defense in depth: hiding the nav entry is not access control - re-check here.
    if not is_admin(user):
        st.error("403 - Admins only")
        st.stop()

    st.title("\\U0001f6e1 Admin Dashboard")
    tab_usage, tab_tenants, tab_audit, tab_dlp = st.tabs(
        ["Usage", "Tenants", "Audit Log", "DLP"])

    with tab_usage:
        st.subheader("Spend by model (last 30 days)")
        try:
            df = _usage_by_model()
            if df.empty:
                st.info(f"No usage rows yet in `{BQ_DATASET}.spend_logs`.")
            else:
                total = float(df["cost_usd"].sum())
                c1, c2, c3 = st.columns(3)
                c1.metric("Requests", f"{int(df['requests'].sum()):,}")
                c2.metric("Cost", f"${total:,.2f}")
                c3.metric("Cost (INR)", f"₹{total * 85:,.0f}")
                import plotly.express as px
                st.plotly_chart(
                    px.bar(df, x="model", y="cost_usd", title="Cost by model (USD)"),
                    use_container_width=True)
                st.dataframe(df, use_container_width=True)
        except Exception as e:
            st.info(f"Usage source not available yet: {e}")

    with tab_tenants:
        st.subheader("Tenants & virtual-key spend")
        try:
            import httpx
            import pandas as pd
            base = os.environ["LITELLM_BASE_URL"].rstrip("/")
            r = httpx.get(
                f"{base}/spend/tags",
                headers={"Authorization": f"Bearer {os.environ.get('LITELLM_MASTER_KEY', '')}"},
                timeout=15)
            r.raise_for_status()
            st.dataframe(pd.DataFrame(r.json()), use_container_width=True)
            st.caption("Per-tenant spend from the LiteLLM gateway (tag-scoped virtual keys).")
        except Exception as e:
            st.info(
                "Wire this to the LiteLLM admin API (`/spend/tags`, `/key/info`) for "
                f"per-tenant budgets and caps. ({e})")

    with tab_audit:
        st.subheader("Recent audit events")
        try:
            df = _recent_audit()
            if df.empty:
                st.info(f"No events in Firestore `{AUDIT_COLLECTION}` yet.")
            else:
                st.dataframe(df, use_container_width=True, height=420)
        except Exception as e:
            st.info(f"Audit collection not available yet: {e}")

    with tab_dlp:
        st.subheader("DLP findings - PII & RESTRICTED replies (last 30 days)")
        try:
            df = _dlp_findings()
            if df.empty:
                st.info("No DLP findings recorded in the last 30 days.")
            else:
                c1, c2 = st.columns(2)
                c1.metric("Findings", f"{int(df['findings'].sum()):,}")
                c2.metric("Redacted", f"{int(df['redacted'].sum()):,}")
                st.dataframe(df, use_container_width=True)
        except Exception as e:
            st.info(
                "DLP findings feed not provisioned yet "
                f"(Cloud DLP inspection results -> `{BQ_DATASET}.dlp_findings`). ({e})")'''
with open('admin_dashboard.py', 'w') as f:
    f.write(ADMIN_PY)
print('admin_dashboard.py written')

## Cell 10: Dockerfile + Cloud Run deployment


In [ ]:
DOCKERFILE = '''
# syntax=docker/dockerfile:1.7
FROM python:3.12-slim

RUN apt-get update && apt-get install -y --no-install-recommends \\
      tini curl ca-certificates build-essential \\
    && rm -rf /var/lib/apt/lists/*

RUN useradd --create-home --shell /bin/bash --uid 10001 app
WORKDIR /app

COPY requirements.txt .
RUN pip install --no-cache-dir --upgrade pip \\
 && pip install --no-cache-dir -r requirements.txt

COPY --chown=app:app . .
USER app

ENV PORT=8080 \\
    STREAMLIT_SERVER_PORT=8080 \\
    STREAMLIT_SERVER_ADDRESS=0.0.0.0 \\
    STREAMLIT_SERVER_HEADLESS=true \\
    STREAMLIT_SERVER_ENABLE_CORS=false \\
    STREAMLIT_SERVER_ENABLE_XSRF_PROTECTION=false \\
    STREAMLIT_BROWSER_GATHER_USAGE_STATS=false \\
    PYTHONUNBUFFERED=1

EXPOSE 8080
HEALTHCHECK --interval=30s --timeout=5s --start-period=15s --retries=3 \\
    CMD curl -fsS http://localhost:8080/_stcore/health || exit 1

ENTRYPOINT ["/usr/bin/tini", "--"]
CMD ["streamlit","run","app.py", \\
     "--server.port=8080","--server.address=0.0.0.0", \\
     "--server.headless=true", \\
     "--server.enableCORS=false","--server.enableXsrfProtection=false"]
'''
with open('Dockerfile', 'w') as f:
    f.write(DOCKERFILE)

DEPLOY = '''
gcloud run deploy documind-ui \\
  --image=us-central1-docker.pkg.dev/$PROJECT/documind/ui:$GIT_SHA \\
  --region=us-central1 --platform=managed \\
  --no-allow-unauthenticated \\
  --ingress=internal-and-cloud-load-balancing \\
  --memory=2Gi --cpu=2 --concurrency=80 --timeout=3600 \\
  --min-instances=1 --max-instances=10 \\
  --cpu-boost --session-affinity --execution-environment=gen2 \\
  --service-account=documind-ui-sa@$PROJECT.iam.gserviceaccount.com \\
  --set-env-vars="AUTH_MODE=iap,LITELLM_BASE_URL=https://litellm.internal,SPEECH_REGION=asia-south1,GOOGLE_CLOUD_PROJECT=documind-ai-YOUR-ID,GOOGLE_CLOUD_LOCATION=us-central1,UPLOAD_BUCKET=documind-ai-YOUR-ID-uploads,LAYOUT_PROCESSOR=projects/documind-ai-YOUR-ID/locations/us/processors/YOUR_LAYOUT_PROC,TTS_CACHE_BUCKET=documind-ai-YOUR-ID-tts-cache,VECTOR_INDEX_ENDPOINT=YOUR_INDEX_ENDPOINT,DEPLOYED_INDEX_ID=documind_deployed,CHUNKS_COLLECTION=chunks,AUDIT_COLLECTION=audit_events,OBSERVABILITY_DATASET=documind_observability,RAG_MODEL=gemini-3.6-flash,ADMIN_EMAILS=admin@documind.ai,ADMIN_DOMAINS=documind.ai,IAP_AUDIENCE=YOUR_IAP_AUDIENCE" \\
  --set-secrets="LITELLM_MASTER_KEY=litellm-master-key:latest,COOKIE_SECRET=cookie-secret:latest" \\
  --vpc-connector=projects/$PROJECT/locations/us-central1/connectors/documind-vpc \\
  --vpc-egress=private-ranges-only

# Grant self-impersonation for V4 signed URLs
gcloud iam service-accounts add-iam-policy-binding \\
  documind-ui-sa@$PROJECT.iam.gserviceaccount.com \\
  --member="serviceAccount:documind-ui-sa@$PROJECT.iam.gserviceaccount.com" \\
  --role="roles/iam.serviceAccountTokenCreator"

# Enable IAP 1-click
gcloud beta run services update documind-ui --region=us-central1 --iap

# Grant user access
gcloud beta iap web add-iam-policy-binding \\
  --resource-type=cloud-run --service=documind-ui --region=us-central1 \\
  --member="user:alice@documind.ai" \\
  --role=roles/iap.httpsResourceAccessor
'''
print('Dockerfile written')
print(DEPLOY)


## Cell 11: Complete file listing + test checklist


In [ ]:
FILES = {
    'app.py': 'Main entry with auth gate + navigation',
    'chat.py': 'Streaming chat + stop button + cost sidebar',
    'documents.py': 'Upload + Doc AI Layout Parser + Vector Search',
    'citations.py': 'render_with_citations + signed_url',
    'voice.py': 'Chirp 3 STT + Chirp 3: HD TTS with SHA-256 cache',
    'auth.py': 'st.login() dev + IAP JWT verification prod',
    'rag.py': 'Query embedding + ANN search + rerank',
    'admin_dashboard.py': 'Usage / Tenants / Audit Log / DLP',
    'Dockerfile': 'python:3.12-slim + tini + healthcheck',
    'requirements.txt': 'Pinned April 2026 versions',
    '.streamlit/config.toml': 'Server + theme config',
}
print('COMPLETE FILE LISTING:')
for f, desc in FILES.items():
    print(f'  {f:30} - {desc}')
print()
print('TEST CHECKLIST:')
checklist = [
    '/_stcore/health returns 200',
    'st.login redirects to Google OAuth consent',
    'st.user.email populated after login',
    'Chat streams token-by-token (not batch)',
    'Stop button interrupts mid-stream cleanly',
    'Cost sidebar updates after each turn',
    'File upload >10MB completes (resumable)',
    'Uploaded PDF renders in streamlit-pdf-viewer',
    'Citation [1] pill renders with hover tooltip',
    'Click pill scrolls to source anchor',
    'Jump to page opens PDF at correct page',
    'Mic button records and transcribes (en-IN + hi-IN)',
    'TTS plays assistant reply (autoplay works after user gesture)',
    'Cache hit on repeated TTS phrase (p95 < 100ms)',
    'Admin tab visible only to admin emails',
    'Non-admin sees 403 on admin page access attempt',
    'IAP JWT verification rejects forged emails',
    'Tier badge renders (green/amber/red)',
    'Compliance banner shows for RESTRICTED replies',
    'LiteLLM 429 BudgetExceededError shown gracefully',
    'WebSocket survives 60-min idle (heartbeat ack)',
    'Session affinity cookie persists across page reloads',
]
for i, item in enumerate(checklist, 1):
    print(f'  [{i:2}] {item}')


## Done!
Complete Streamlit DocuMind frontend:

**Core files**: app.py (entry) + chat.py (streaming) + documents.py (upload) + voice.py (Chirp 3) + auth.py (IAP + st.login) + admin_dashboard.py

**Deployment**: Dockerfile + gcloud run deploy with --session-affinity + --cpu-boost + --timeout=3600 + IAP 1-click + self-impersonation for V4 signed URLs

**Production levers**: session affinity + XSRF-off + stream_options for authoritative usage + prompt-embedded [N] citations + SHA-256 TTS cache + verified April 2026 pricing
